In [4]:
import pandas as pd
import numpy as np

In [5]:
pd.set_option('display.max_columns', None)

In [6]:
df = pd.read_csv("../data/cleaned_zomato.csv")

In [7]:
df.head()

,name,online_order,book_table,rate,votes,approx_cost,restaurant_type
0,Jalsa,Yes,Yes,4.1,775,800,Buffet
1,Spice Elephant,Yes,No,4.1,787,800,Buffet
2,San Churro Cafe,Yes,No,3.8,918,800,Buffet
3,Addhuri Udupi Bhojana,No,No,3.7,88,300,Buffet
4,Grand Village,No,No,3.8,166,600,Buffet


In [8]:
df.shape

(148, 7)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 148 entries, 0 to 147
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             148 non-null    str    
 1   online_order     148 non-null    str    
 2   book_table       148 non-null    str    
 3   rate             148 non-null    float64
 4   votes            148 non-null    int64  
 5   approx_cost      148 non-null    int64  
 6   restaurant_type  148 non-null    str    
dtypes: float64(1), int64(2), str(4)
memory usage: 8.2 KB


In [10]:
df.isnull().sum()

name               0
online_order       0
book_table         0
rate               0
votes              0
approx_cost        0
restaurant_type    0
dtype: int64

In [11]:
print(df.columns.tolist())

['name', 'online_order', 'book_table', 'rate', 'votes', 'approx_cost', 'restaurant_type']


In [12]:
#Step 4: Creating Combined Features
df["features"] = (
    df["restaurant_type"].astype(str) + " " +
    df["online_order"].astype(str) + " " +
    df["book_table"].astype(str) + " " +
    df["approx_cost"].astype(str) + " " +
    df["rate"].astype(str)
)

df[["name", "features"]].head()

,name,features
0,Jalsa,Buffet Yes Yes 800 4.1
1,Spice Elephant,Buffet Yes No 800 4.1
2,San Churro Cafe,Buffet Yes No 800 3.8
3,Addhuri Udupi Bhojana,Buffet No No 300 3.7
4,Grand Village,Buffet No No 600 3.8


In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [14]:
tfidf = TfidfVectorizer()

In [15]:
tfidf_matrix = tfidf.fit_transform(df["features"])

In [1]:
print(tfidf_matrix.shape)

NameError: name 'tfidf_matrix' is not defined

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

In [18]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [19]:
print(similarity_matrix.shape)

(148, 148)


In [20]:
similarity_matrix

array([[1.        , 0.94412723, 0.94412723, ..., 0.23574584, 0.43233025,
        0.27787454],
       [0.94412723, 1.        , 1.        , ..., 0.17540663, 0.58625586,
        0.20675248],
       [0.94412723, 1.        , 1.        , ..., 0.17540663, 0.58625586,
        0.20675248],
       ...,
       [0.23574584, 0.17540663, 0.17540663, ..., 1.        , 0.19134407,
        0.330651  ],
       [0.43233025, 0.58625586, 0.58625586, ..., 0.19134407, 1.        ,
        0.225538  ],
       [0.27787454, 0.20675248, 0.20675248, ..., 0.330651  , 0.225538  ,
        1.        ]], shape=(148, 148))

In [27]:
# Create an index using restaurant names
indices = pd.Series(df.index, index=df["name"]).drop_duplicates()

indices.head()

name
Jalsa                    0
Spice Elephant           1
San Churro Cafe          2
Addhuri Udupi Bhojana    3
Grand Village            4
dtype: int64

In [31]:
def recommend_restaurants(restaurant_name, top_n=5):

    # Check if restaurant exists
    if restaurant_name not in indices:
        return "Restaurant not found."

    # Get restaurant index
    idx = indices[restaurant_name]

    # Get similarity scores
    similarity_scores = list(enumerate(similarity_matrix[idx]))

    # Sort by similarity
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the restaurant itself
    similarity_scores = similarity_scores[1:top_n+1]

    # Get recommended restaurant indices
    restaurant_indices = [i[0] for i in similarity_scores]

    # Create result DataFrame
    result = df.iloc[restaurant_indices][
        ["name", "restaurant_type", "rate", "approx_cost", "votes"]
    ]

    # Sort by rating and votes
    result = result.sort_values(
        by=["rate", "votes"],
        ascending=False
    )

    # Rename columns
    result = result.rename(columns={
        "name": "Restaurant",
        "restaurant_type": "Type",
        "rate": "Rating",
        "approx_cost": "Cost for Two",
        "votes": "Votes"
    })

    return result

In [32]:
recommend_restaurants("Jalsa")

,Restaurant,Type,Rating,Cost for Two,Votes
1,Spice Elephant,Buffet,4.1,800,787
43,Domino's Pizza,Dining,3.9,800,540
2,San Churro Cafe,Buffet,3.8,800,918
61,Goa 0 Km,Dining,3.6,800,163
6,Rosewood International Hotel - Bar & Restaurant,Buffet,3.6,800,8


In [33]:
recommend_restaurants("Spice Elephant")

,Restaurant,Type,Rating,Cost for Two,Votes
0,Jalsa,Buffet,4.1,800,775
43,Domino's Pizza,Dining,3.9,800,540
2,San Churro Cafe,Buffet,3.8,800,918
61,Goa 0 Km,Dining,3.6,800,163
6,Rosewood International Hotel - Bar & Restaurant,Buffet,3.6,800,8
